# Step 3 — Gold: Customer Value by Segment

**Purpose:** Build business aggregates on the Silver table. Because quality was enforced on
input and the data is already auto-clustered, Gold reads trusted, well-laid-out data — no
defensive filtering, no pre-aggregation `OPTIMIZE` job.

### What This Notebook Does
1. Reads `shift_left_optimization.silver.customers_silver` (validated + auto-clustered).
2. Aggregates customer value by `membership_type` and `city`, written with `CLUSTER BY AUTO`.
3. Demonstrates Delta **Time Travel** for auditability.

### Source & Target
| Direction | Table |
|-----------|-------|
| Source | `shift_left_optimization.silver.customers_silver` |
| Target (Gold Delta) | `shift_left_optimization.gold.segment_value` |

> **Prerequisites:** Run `02_customers_silver` first.

---

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
SILVER_TABLE = 'shift_left_optimization.silver.customers_silver'
GOLD_TABLE   = 'shift_left_optimization.gold.segment_value'

spark.sql('CREATE SCHEMA IF NOT EXISTS shift_left_optimization.gold')
spark.sql(f'DROP TABLE IF EXISTS {GOLD_TABLE}')

print(f'Source: {SILVER_TABLE}')
print(f'Target: {GOLD_TABLE}')

In [0]:
from pyspark.sql import functions as F

---

### Read the trusted, auto-clustered Silver table

In [0]:
silver_df = spark.read.table(SILVER_TABLE)

print(f'Silver rows: {silver_df.count()}')

---

### Aggregate customer value by membership tier and city

In [0]:
gold_df = (
    silver_df
    .groupBy('membership_type', 'city')
    .agg(
        F.count('*').alias('customer_count'),
        F.round(F.avg('total_spend'), 2).alias('avg_total_spend'),
        F.round(F.avg('avg_rating'), 2).alias('avg_rating'),
        F.round(F.avg('days_since_last_purchase'), 1).alias('avg_days_since_purchase'),
    )
)

---

### Write to Gold + enable automatic clustering

In [0]:
(gold_df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(GOLD_TABLE))

spark.sql(f'ALTER TABLE {GOLD_TABLE} CLUSTER BY AUTO')
print(f'Gold rows written: {spark.read.table(GOLD_TABLE).count()}')

---

### Validation + Time Travel

In [0]:
%sql
SELECT membership_type, city, customer_count, avg_total_spend, avg_rating
FROM shift_left_optimization.gold.segment_value
ORDER BY avg_total_spend DESC
LIMIT 10

In [0]:
%sql
DESCRIBE HISTORY shift_left_optimization.gold.segment_value

In [0]:
# Time Travel: read the first version (versionAsOf 0)
gold_v0 = (
    spark.read.format('delta')
    .option('versionAsOf', 0)
    .table(GOLD_TABLE)
)
print(f'Rows in version 0: {gold_v0.count()}')